# 01 – Load and combine

Load the 24 monthly Bristol City Council "spend over £500" files (April 2024 – March 2026) and combine them into one dataset.

In [1]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
files = sorted(RAW.glob("spend_*.csv"))
print(len(files), "files found\n")

for f in files:
    df = pd.read_csv(f, encoding="cp1252", nrows=5)
    print(f.name, "|", len(df.columns), "columns |", list(df.columns))

24 files found

spend_2024_04.csv | 9 columns | ['Body', 'Body Name', 'Name', 'amount', 'Pay Date', 'Transaction Number', 'Description 1 ', 'Description 2 ', 'Description 3 ']
spend_2024_05.csv | 9 columns | ['Body', 'Body Name', 'Name', 'amount', 'Pay Date', 'Transaction Number', 'Description 1 ', 'Description 2 ', 'Description 3 ']
spend_2024_06.csv | 9 columns | ['Body', 'Body Name', 'Name', 'amount', 'Pay Date', 'Transaction Number', 'Description 1 ', 'Description 2 ', 'Description 3 ']
spend_2024_07.csv | 9 columns | ['Body', 'Body Name', 'Name', 'amount', 'Pay Date', 'Transaction Number', 'Description 1 ', 'Description 2 ', 'Description 3 ']
spend_2024_08.csv | 9 columns | ['Body', 'Body Name', 'Name', 'amount', 'Pay Date', 'Transaction Number', 'Description 1 ', 'Description 2 ', 'Description 3 ']
spend_2024_09.csv | 9 columns | ['Body', 'Body Name', 'Name', 'amount', 'Pay Date', 'Transaction Number', 'Description 1 ', 'Description 2 ', 'Description 3 ']
spend_2024_10.csv | 9 co

## Combine the files

The files aren't fully consistent:
- Some column names have trailing spaces (`"Description 1 "` vs `"Description 1"`).
- October and November 2025 are saved as UTF-8; the rest are Windows-1252.
- November 2025 has its columns in a different order, with `Amount` capitalised.

So each file is read with the right encoding, column names are cleaned, and columns are matched by name. Each file also ends with a total row (no supplier, no date), which is removed here and kept for checking totals later.

In [2]:
def read_spend_file(path):
    # Try UTF-8 first; if that fails, the file is Windows-1252
    try:
        df = pd.read_csv(path, encoding="utf-8-sig", dtype=str)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="cp1252", dtype=str)
    df.columns = df.columns.str.strip().str.lower()
    df["source_file"] = path.name
    return df

frames = [read_spend_file(f) for f in files]
raw = pd.concat(frames, ignore_index=True)

raw = raw.rename(columns={
    "body": "council",
    "body name": "body_url",
    "name": "supplier",
    "pay date": "pay_date",
    "transaction number": "transaction_number",
    "description 1": "expense_type",
    "description 2": "service_area",
    "description 3": "project",
})

# Remove completely empty rows
data_cols = [c for c in raw.columns if c != "source_file"]
raw = raw.dropna(how="all", subset=data_cols)

# Total rows: no supplier, no date, but an amount
is_total = raw["supplier"].isna() & raw["pay_date"].isna() & raw["amount"].notna()
total_rows = raw[is_total]
spend = raw[~is_total].copy()

print("Columns:", list(spend.columns), "\n")
print(f"Payment lines: {len(spend):,}")
print(f"Total rows removed: {len(total_rows)}\n")
print(spend.groupby("source_file").size().to_string())

Columns: ['council', 'body_url', 'supplier', 'amount', 'pay_date', 'transaction_number', 'expense_type', 'service_area', 'project', 'source_file'] 

Payment lines: 141,930
Total rows removed: 23

source_file
spend_2024_04.csv    6726
spend_2024_05.csv    6573
spend_2024_06.csv    5380
spend_2024_07.csv    5944
spend_2024_08.csv    6076
spend_2024_09.csv    4983
spend_2024_10.csv    5867
spend_2024_11.csv    5897
spend_2024_12.csv    5411
spend_2025_01.csv    5724
spend_2025_02.csv    5616
spend_2025_03.csv    6692
spend_2025_04.csv    7196
spend_2025_05.csv    5295
spend_2025_06.csv    5268
spend_2025_07.csv    6076
spend_2025_08.csv    5474
spend_2025_09.csv    5543
spend_2025_10.csv    6390
spend_2025_11.csv    4654
spend_2025_12.csv    5860
spend_2026_01.csv    5783
spend_2026_02.csv    6327
spend_2026_03.csv    7175


In [3]:
print(len(spend.columns), "columns\n")

# Which file had no total row removed?
with_total = set(total_rows["source_file"])
print("File with no total row removed:", [f.name for f in files if f.name not in with_total], "\n")

# Any leftover rows that look like totals?
looks_like_total = spend[spend["supplier"].str.contains("total", case=False, na=False) | spend["pay_date"].isna()]
print(looks_like_total[["source_file", "supplier", "amount", "pay_date"]].to_string())

10 columns

File with no total row removed: ['spend_2025_06.csv'] 

              source_file                                              supplier      amount    pay_date
6274    spend_2024_04.csv                             Total Merchandise Limited      389.94  05/04/2024
6275    spend_2024_04.csv                             Total Merchandise Limited    1,420.28  05/04/2024
6276    spend_2024_04.csv                             Total Merchandise Limited      475.97  05/04/2024
6277    spend_2024_04.csv                             Total Merchandise Limited      807.17  11/04/2024
12997   spend_2024_05.csv                             Total Merchandise Limited      957.02  15/05/2024
12998   spend_2024_05.csv                             Total Merchandise Limited    1,112.00  29/05/2024
12999   spend_2024_05.csv                             Total Merchandise Limited      461.23  31/05/2024
18407   spend_2024_06.csv                             Total Merchandise Limited      779.56  14/06/2

In [4]:
june = spend[spend["source_file"] == "spend_2025_06.csv"]
print("Last 3 rows of June 2025:")
print(june[["supplier", "amount", "pay_date", "transaction_number"]].tail(3).to_string(), "\n")

no_date = spend[spend["pay_date"].isna()]
print(f"Rows with no pay date: {len(no_date)}")
print(no_date[["source_file", "supplier", "amount", "transaction_number"]].head(10).to_string())

Last 3 rows of June 2025:
              supplier    amount    pay_date transaction_number
88659  Ward Solicitors         0  27/06/2025            6067943
88660    Wren Kitchens  8,319.73  17/06/2025            6067793
88661    Wren Kitchens  1,663.95  17/06/2025            6067793 

Rows with no pay date: 0
Empty DataFrame
Columns: [source_file, supplier, amount, transaction_number]
Index: []


## Checks

- All 24 files loaded: **141,930 payment lines**, 10 columns.
- 23 total rows removed. June 2025's file has no total row; its last rows are normal payments.
- Some real suppliers have "Total" in their name (e.g. Total Merchandise Limited), so total rows are found by "no supplier and no date", not by the word "Total".
- The amount format varies between files (some have a £ sign, some don't). This is fixed in `03_clean.ipynb`.

In [5]:
PROCESSED = Path("../data/processed")

spend.to_csv(PROCESSED / "spend_combined.csv", index=False, encoding="utf-8")
total_rows[["source_file", "amount"]].to_csv(PROCESSED / "file_totals.csv", index=False, encoding="utf-8")

print(f"Saved spend_combined.csv ({len(spend):,} rows)")
print(f"Saved file_totals.csv ({len(total_rows)} rows)")

Saved spend_combined.csv (141,930 rows)
Saved file_totals.csv (23 rows)
